In [8]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Load data
messages = pd.read_csv("/Users/aryan/ragebait-bot/data/raw/messages.csv")
labels = pd.read_csv("/Users/aryan/ragebait-bot/data/annotated/labels.csv")

# Average scores across annotators
avg_labels = labels.groupby("message_id")["rage_score"].mean().reset_index()

# Merge text + labels
data = pd.merge(messages, avg_labels, on="message_id")

# Features (X) and target (y)
X_text = data["text"]
y = (data["rage_score"] >= 0.6).astype(int)

# Convert text → numbers
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(X_text)

# Train model
model = LogisticRegression()
model.fit(X, y)

# Test on same data (for now)
preds = model.predict_proba(X)[:, 1]

# Print results
for i in range(len(data)):
    print(f"Text: {data['text'][i]}")
    print(f"Actual: {y[i]:.2f} | Predicted: {preds[i]:.2f}")
    print("-" * 50)

Text: "Michael Jackson did not have talent"
Actual: 1.00 | Predicted: 0.64
--------------------------------------------------
Text: "if you talk about him so much why don't you marry him"
Actual: 0.00 | Predicted: 0.49
--------------------------------------------------
Text: "Glad that that fucking dickhead died"
Actual: 1.00 | Predicted: 0.62
--------------------------------------------------
Text: "You're too stupid to even understand"
Actual: 1.00 | Predicted: 0.70
--------------------------------------------------
Text: "This is a stupid question"
Actual: 1.00 | Predicted: 0.62
--------------------------------------------------
Text: "straight men count as lesbian and straight women count as gay"
Actual: 1.00 | Predicted: 0.67
--------------------------------------------------
Text: "You guys actually need a life. Why do you ruin peoples days by using this stupid gen z thing called ragebait"
Actual: 0.00 | Predicted: 0.48
--------------------------------------------------
Text: "Ro

In [10]:
def predict_message(text):
    # Convert text using SAME vectorizer
    X_new = vectorizer.transform([text])

    # Get probability
    prob = model.predict_proba(X_new)[0][1]

    return prob

In [12]:
test_msg = "Everyone who likes this is an idiot"

score = predict_message(test_msg)

print("\n--- NEW MESSAGE ---")
print(test_msg)
print(f"Rage-bait probability: {score:.2f}")


--- NEW MESSAGE ---
Everyone who likes this is an idiot
Rage-bait probability: 0.46


In [ ]:
while True:
    user_input = input("\nEnter a message (or type 'exit'): ")

    if user_input.lower() == "exit":
        break

    score = predict_message(user_input)

    print(f"Rage-bait probability: {score:.2f}")

    if score > 0.7:
        print("⚠️ Possible rage bait detected 😄")


Enter a message (or type 'exit'):  can you send me notes


Rage-bait probability: 0.47



Enter a message (or type 'exit'):  you are easy to draw


Rage-bait probability: 0.69



Enter a message (or type 'exit'):  you look like the type of person to like hitler


...

Rage-bait probability: 0.56



Enter a message (or type 'exit'):  based


Rage-bait probability: 0.58



Enter a message (or type 'exit'):  cooked


Rage-bait probability: 0.56
